# Transformer Model

This notebook starts the transformer-based part of the project.

The goal is to understand the main ideas before training a full model:

- what a tokenizer does
- what a pre-trained model is
- why DistilBERT is useful
- how the dataset must be prepared for fine-tuning

## Step 1: Load The Dataset

We use the same dataset from the baseline model so we can compare both approaches later.

In [4]:
from datasets import load_dataset

dataset = load_dataset("dair-ai/emotion")
label_names = dataset["train"].features["label"].names

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

## Step 2: Choose A Pre-Trained Model

A transformer model is usually not trained from zero for a small project.

Instead, we start from a model that already learned general language patterns from a large amount of text.

For this project, we will use `distilbert-base-uncased`.

DistilBERT is a smaller and faster version of BERT, which makes it a good first transformer model to learn with.

In [5]:
model_checkpoint = "distilbert-base-uncased"
model_checkpoint

'distilbert-base-uncased'

## Step 3: Understand The Tokenizer

A tokenizer converts text into tokens that the transformer model can understand.

In the baseline model, TF-IDF converted text into word-based numeric features.

For transformers, the tokenizer converts text into token IDs.

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

c:\Users\Admin\OneDrive\Python\emotion-classification-transformers\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

## Step 4: Tokenize One Example

Before tokenizing the full dataset, it is useful to inspect one example.

In [7]:
example = dataset["train"][0]
example

{'text': 'i didnt feel humiliated', 'label': 0}

In [8]:
tokens = tokenizer(example["text"])
tokens

{'input_ids': [101, 1045, 2134, 2102, 2514, 26608, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

The tokenizer output includes:

- `input_ids`: numeric IDs representing tokens
- `attention_mask`: tells the model which tokens are real text and which tokens are padding

In [9]:
tokenizer.convert_ids_to_tokens(tokens["input_ids"])

['[CLS]', 'i', 'didn', '##t', 'feel', 'humiliated', '[SEP]']

## Step 5: Tokenize The Dataset

Now we create a function that tokenizes batches of text.

We use padding and truncation so the model receives inputs in a consistent format.

In [10]:
def tokenize_batch(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)


tokenized_dataset = dataset.map(tokenize_batch, batched=True)
tokenized_dataset

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
})

## Step 6: What Fine-Tuning Means

Fine-tuning means taking a pre-trained model and training it a little more on a specific task.

In this project, DistilBERT already understands general English patterns, but it does not yet know our specific emotion labels.

Fine-tuning teaches the model to connect text patterns with the six emotion classes.

## Next Step

The next notebook section will train DistilBERT for emotion classification and compare it with the baseline model.

Baseline result to compare later:

```text
TF-IDF + Logistic Regression accuracy: 0.8615
```